# 02 - Trajectories: overlay, box plot, results table

Reads only `results/` (never re-runs SLAM). Run `python -m pipeline.evaluate` first.
Outputs go to `paper/figures/` and `paper/tables/` in the shared IEEE style.

In [ ]:
import sys, copy
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from evo.core import sync
from evo.tools import file_interface

from pipeline.common import FIGURES_DIR, TABLES_DIR, RESULTS_DIR, load_sequences, load_systems, load_system_config

plt.style.use(ROOT / "paper" / "ieee.mplstyle")
FIGURES_DIR.mkdir(parents=True, exist_ok=True); TABLES_DIR.mkdir(parents=True, exist_ok=True)
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
summary = pd.read_csv(RESULTS_DIR / "summary.csv")
summary

## Trajectory overlay (required figure): all systems + ground truth, one representative run (median ATE)

In [ ]:
SEQUENCE = load_sequences()["sequences"][0]["name"]   # change to the sequence shown in the paper
seq_cfg = next(s for s in load_sequences()["sequences"] if s["name"] == SEQUENCE)
gt = file_interface.read_tum_trajectory_file(str(ROOT / seq_cfg["ground_truth"]))

fig, ax = plt.subplots()
ax.plot(gt.positions_xyz[:, 0], gt.positions_xyz[:, 1], color="0.6", linestyle="-", marker="", lw=2, label="ground truth")
for system in load_systems():
    runs = metrics[(metrics.slug == system["slug"]) & (metrics.sequence == SEQUENCE) & metrics.success]
    if runs.empty:
        continue
    median_run = runs.iloc[(runs.ate_rmse_m - runs.ate_rmse_m.median()).abs().argmin()]["run"]
    est = file_interface.read_tum_trajectory_file(str(RESULTS_DIR / system["slug"] / SEQUENCE / median_run / "trajectory_tum.txt"))
    ref, est = sync.associate_trajectories(gt, est, max_diff=load_sequences()["evaluation"]["t_max_diff"])
    est.align(ref, correct_scale=load_system_config(system["slug"])["alignment"] == "sim3")
    ax.plot(est.positions_xyz[:, 0], est.positions_xyz[:, 1], markevery=max(1, est.num_poses // 12), label=system["system"])
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_aspect("equal", adjustable="datalim")
ax.legend(loc="best")
fig.savefig(FIGURES_DIR / "trajectory_overlay.pdf")

## ATE RMSE over N runs (box plot, single runs as dots)

In [ ]:
ok = metrics[metrics.success]
fig, axes = plt.subplots(1, ok.sequence.nunique(), figsize=(7.16, 2.4), sharey=True, squeeze=False)
for ax, (seq, d) in zip(axes[0], ok.groupby("sequence")):
    systems = sorted(d.system.unique())
    ax.boxplot([d[d.system == s].ate_rmse_m for s in systems], tick_labels=systems, widths=0.5)
    for i, s in enumerate(systems, start=1):
        ax.plot([i] * (d.system == s).sum(), d[d.system == s].ate_rmse_m, "o", color="k", ms=2, ls="")
    ax.set_title(seq); ax.tick_params(axis="x", rotation=30)
axes[0][0].set_ylabel("ATE RMSE [m]")
fig.savefig(FIGURES_DIR / "ate_boxplot.pdf")

## Results table -> LaTeX (median (IQR), success k/N)

In [ ]:
tab = summary.copy()
tab["ATE RMSE [m]"] = tab.apply(lambda r: f"{r.ate_rmse_median_m:.3f} ({r.ate_rmse_iqr_m:.3f})" if r.successes else "--", axis=1)
tab["success"] = tab.apply(lambda r: f"{int(r.successes)}/{int(r.runs)}", axis=1)
pivot = tab.pivot(index="system", columns="sequence", values=["ATE RMSE [m]", "success"])
(TABLES_DIR / "results.tex").write_text(pivot.to_latex(escape=True))
pivot